# ChatPPT Homework v0.4

**Part 1**：将 ChatPPT 以 **HTTPS 服务**形式发布（`IP:PORT` 方式访问），通过自签名证书实现加密传输

### 2.1 安装依赖

In [ ]:
# 安装生成自签名证书所需的 cryptography 库（如已安装可跳过）
%pip install cryptography --quiet
print('✅ 依赖安装完成')

### 2.2 生成自签名 SSL 证书

In [9]:
import os
import sys
sys.path.insert(0, '../src')
import socket
import datetime
import ipaddress
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa

CERT_DIR  = '../certs'
CERT_FILE = os.path.join(CERT_DIR, 'cert.pem')
KEY_FILE  = os.path.join(CERT_DIR, 'key.pem')
SERVER_PORT = 7888

os.makedirs(CERT_DIR, exist_ok=True)

# ── 获取本机局域网 IP ───────────────────────────────────────────
def get_local_ip() -> str:
    """通过连接外部地址（不发送数据）探测本机出口 IP。"""
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
            s.connect(('8.8.8.8', 80))
            return s.getsockname()[0]
    except Exception:
        return '127.0.0.1'

LOCAL_IP = get_local_ip()
print(f'本机 IP：{LOCAL_IP}')

# ── 生成 RSA 私钥 ───────────────────────────────────────────────
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)

# ── 构建证书主体信息 ────────────────────────────────────────────
subject = issuer = x509.Name([
    x509.NameAttribute(NameOID.COUNTRY_NAME, 'CN'),
    x509.NameAttribute(NameOID.ORGANIZATION_NAME, 'ChatPPT'),
    x509.NameAttribute(NameOID.COMMON_NAME, LOCAL_IP),
])

# ── SAN：同时支持 IP 和 localhost 访问 ──────────────────────────
san = x509.SubjectAlternativeName([
    x509.IPAddress(ipaddress.IPv4Address(LOCAL_IP)),
    x509.IPAddress(ipaddress.IPv4Address('127.0.0.1')),
    x509.DNSName('localhost'),
])

# ── 签发自签名证书（有效期 365 天）─────────────────────────────
cert = (
    x509.CertificateBuilder()
    .subject_name(subject)
    .issuer_name(issuer)
    .public_key(private_key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(datetime.datetime.utcnow())
    .not_valid_after(datetime.datetime.utcnow() + datetime.timedelta(days=365))
    .add_extension(san, critical=False)
    .sign(private_key, hashes.SHA256())
)

# ── 写入文件 ────────────────────────────────────────────────────
with open(KEY_FILE, 'wb') as f:
    f.write(private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.TraditionalOpenSSL,
        encryption_algorithm=serialization.NoEncryption(),
    ))

with open(CERT_FILE, 'wb') as f:
    f.write(cert.public_bytes(serialization.Encoding.PEM))

print(f'✅ SSL 证书已生成')
print(f'   证书：{os.path.abspath(CERT_FILE)}')
print(f'   私钥：{os.path.abspath(KEY_FILE)}')
print(f'\n🌐 服务启动后访问地址：https://{LOCAL_IP}:{SERVER_PORT}')

本机 IP：10.1.105.25
✅ SSL 证书已生成
   证书：c:\Users\Elon_Sun\Code\github\ChatPPT\certs\cert.pem
   私钥：c:\Users\Elon_Sun\Code\github\ChatPPT\certs\key.pem

🌐 服务启动后访问地址：https://10.1.105.25:7888


C:\Users\Elon_Sun\AppData\Local\Temp\ipykernel_9256\625577596.py:59: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  .not_valid_before(datetime.datetime.utcnow())
C:\Users\Elon_Sun\AppData\Local\Temp\ipykernel_9256\625577596.py:60: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  .not_valid_after(datetime.datetime.utcnow() + datetime.timedelta(days=365))


### 2.3 初始化 LLM 与 PPTX 生成函数

In [10]:
import re
import importlib
import openai

# 强制重新加载所有相关模块，确保使用最新代码
import data_structures, slide_builder, layout_manager, input_parser, ppt_generator
for _mod in [data_structures, slide_builder, layout_manager, input_parser, ppt_generator]:
    importlib.reload(_mod)

from layout_manager import LayoutManager
from template_manager import load_template, get_layout_mapping
from input_parser import parse_input_text
from ppt_generator import generate_presentation
from data_structures import Slide

# 加载 formatter.txt prompt
with open('../prompts/formatter.txt', encoding='utf-8') as f:
    FORMATTER_PROMPT = f.read()

PPT_TEMPLATE    = '../templates/MasterTemplate.pptx'
_DEFAULT_KEY    = os.environ.get('OPENAI_API_KEY', '')
_DEFAULT_URL    = os.environ.get('OPENAI_BASE_URL', '')


def call_llm(user_input: str, api_key: str, base_url: str = '') -> str:
    """调用 LLM，将自然语言转换为 ChatPPT Markdown 格式。"""
    client = openai.OpenAI(
        api_key=api_key,
        base_url=base_url.strip() or None,
    )
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': FORMATTER_PROMPT},
            {'role': 'user',   'content': user_input},
        ],
        temperature=0.7,
    )
    return resp.choices[0].message.content


def md_to_pptx(markdown_text: str) -> str:
    """将 ChatPPT Markdown 解析为 PPTX 文件，返回生成文件的绝对路径。"""
    prs = load_template(PPT_TEMPLATE)
    layout_map = get_layout_mapping(prs)
    lm = LayoutManager(layout_map)
    ppt_data, title = parse_input_text(markdown_text, lm)
    safe = re.sub(r'[\\/:*?"<>|]', '_', title) or 'ChatPPT_Output'
    path = os.path.abspath(f'../outputs/{safe}.pptx')
    os.makedirs('../outputs', exist_ok=True)
    generate_presentation(ppt_data, PPT_TEMPLATE, path)
    return path


print('✓ LLM 调用与 PPTX 生成函数已定义')

✓ LLM 调用与 PPTX 生成函数已定义


### 2.4 启动 HTTPS Gradio 服务

执行后在浏览器打开 `https://<本机IP>:7888`，首次访问会出现证书警告（自签名证书），点击「高级 → 继续访问」即可。

In [ ]:
import gradio as gr
import traceback


def chat(message: str, history: list, api_key: str, base_url: str):
    if not message.strip():
        yield history, None
        return

    key = api_key.strip() or _DEFAULT_KEY
    if not key:
        yield history + [
            {'role': 'user',      'content': message},
            {'role': 'assistant', 'content': '⚠️ 请先输入 OpenAI API Key。'},
        ], None
        return

    # Step 1：LLM 格式化
    try:
        markdown = call_llm(message, key, base_url)
    except Exception as e:
        yield history + [
            {'role': 'user',      'content': message},
            {'role': 'assistant', 'content': f'❌ LLM 调用失败：{e}'},
        ], None
        return

    # Step 2：生成 PPTX
    pptx_path, status = None, ''
    try:
        pptx_path = md_to_pptx(markdown)
        status = f'✅ PPT 已生成：`{pptx_path}`'
    except Exception as e:
        status = f'⚠️ PPTX 生成失败：{e}\n\n```\n{traceback.format_exc()}\n```'

    response = (
        f'{status}\n\n'
        f'**ChatPPT Markdown：**\n\n'
        f'```markdown\n{markdown}\n```'
    )
    yield history + [
        {'role': 'user',      'content': message},
        {'role': 'assistant', 'content': response},
    ], pptx_path


with gr.Blocks(title='ChatPPT') as demo:
    gr.Markdown('# ChatPPT — AI 驱动的 PPT 生成器')
    gr.Markdown(
        '输入任意主题或内容描述，AI 将自动格式化为 ChatPPT Markdown 并生成 PowerPoint 文件。'
    )

    with gr.Row():
        api_key_box = gr.Textbox(
            label='OpenAI API Key',
            placeholder='sk-...',
            type='password',
            value=_DEFAULT_KEY,
            scale=3,
        )
        base_url_box = gr.Textbox(
            label='API Base URL（可选，留空使用默认）',
            placeholder='https://api.openai.com/v1',
            value=_DEFAULT_URL,
            scale=3,
        )

    chatbot = gr.Chatbot(
        label='对话记录',
        height=420,
    )

    with gr.Row():
        msg_box = gr.Textbox(
            label='输入 PPT 主题或内容',
            placeholder='例如：用 6 页 PPT 介绍人工智能在医疗领域的应用，包括技术原理、应用场景和挑战',
            lines=3,
            scale=5,
        )
        with gr.Column(scale=1, min_width=120):
            submit_btn = gr.Button('生成 PPT', variant='primary')
            clear_btn  = gr.Button('清空对话')

    file_output = gr.File(label='下载 PPT 文件')

    submit_btn.click(
        fn=chat,
        inputs=[msg_box, chatbot, api_key_box, base_url_box],
        outputs=[chatbot, file_output],
    )
    msg_box.submit(
        fn=chat,
        inputs=[msg_box, chatbot, api_key_box, base_url_box],
        outputs=[chatbot, file_output],
    )
    clear_btn.click(lambda: ([], None), outputs=[chatbot, file_output])


# ── HTTPS 启动：监听所有网卡，使用自签名证书 ────────────────────
print(f'\n启动 HTTPS 服务...')
print(f'本机访问：https://127.0.0.1:{SERVER_PORT}')
print(f'局域网访问：https://{LOCAL_IP}:{SERVER_PORT}')
print('（浏览器首次访问会提示证书警告，点击「高级 → 继续访问」即可）\n')

demo.launch(
    server_name='0.0.0.0',        # 监听所有网卡接口，允许局域网访问
    server_port=SERVER_PORT,       # 端口号
    ssl_keyfile=KEY_FILE,          # SSL 私钥
    ssl_certfile=CERT_FILE,        # SSL 证书
    ssl_verify=False,              # 允许自签名证书（不验证 CA 链）
    share=False,
    allowed_paths=[os.path.abspath('../outputs')],
)


启动 HTTPS 服务...
本机访问：https://127.0.0.1:7888
局域网访问：https://10.1.105.25:7888
（浏览器首次访问会提示证书警告，点击「高级 → 继续访问」即可）

* Running on local URL:  https://0.0.0.0:7888
* To create a public link, set `share=True` in `launch()`.


Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "C:\Users\Elon_Sun\Softwares\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Elon_Sun\Softwares\Python313\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "C:\Users\Elon_Sun\Softwares\Python313\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^

2026-04-01 16:15:35 | DEBUG | utils:remove_all_slides:10 - 模板中的幻灯片已被移除。
2026-04-01 16:15:35 | DEBUG | ppt_generator:generate_presentation:96 - 设置幻灯片标题: Claude Code 介绍
2026-04-01 16:15:35 | DEBUG | ppt_generator:generate_presentation:96 - 设置幻灯片标题: 什么是Claude Code？
2026-04-01 16:15:35 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: Claude Code是一个先进的编程语言。
2026-04-01 16:15:35 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: 旨在提高开发效率和代码可读性。
2026-04-01 16:15:35 | DEBUG | ppt_generator:generate_presentation:96 - 设置幻灯片标题: Claude Code的特点
2026-04-01 16:15:35 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: 强类型系统，减少类型错误。
2026-04-01 16:15:35 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: 语法简洁，易于学习和使用。
2026-04-01 16:15:35 | DEBUG | ppt_generator:generate_presentation:96 - 设置幻灯片标题: Claude Code的应用场景
2026-04-01 16:15:35 | DEBUG | ppt_generator:generate_presentation:109 - 添加列表项: Web开发：适合构建动态网页。
2026-04-01 16:15:35 | DEBUG | ppt_generator:generate_presentation:1